-experiment with architecture to see what affected performance. What was essential in a unet
-explain metrics

In [13]:
import pandas as pd

df = pd.read_csv("results.csv")

df = df.drop(columns=["dice", "loss"])

df = df.rename(columns={
    "pixel_acc": "Pixel Accuracy",
    "iou": "IoU",
    "model": "Experiment"
})

f = {
    "base_unet.pth" : "Default U-Net",
    "single_skip_middle_unet.pth": "Only Middle Skip Connection",
    "single_skip_first_unet.pth": "Only Top Skip Connection",
    "bottom_to_top_unet.pth": "Bottom Down Block Connect To Top Up Block",
    "four_blocks_unet.pth": "Extra Block U-Net",
    "no_skip_unet.pth": "U-Net With No Skip Connections",
    "single_skip_bottom_unet.pth": "Only Bottom Skip Connection",
    "top_to_bottom_unet.pth": "Top Down Block Connect To Bottom Up Block",
    "two_blocks_unet.pth" : "Only Two Blocks",
    "bottom_and_middle_skip_unet.pth": "Top Skip Connection Removed",
    "one_block_unet.pth": "Only One Block",
    "one_block_more_capacity_unet.pth": "Only One Block With More Capacity"
}

df["Experiment"] = df["Experiment"].map(f)

df = df.round(4)

df = df.sort_values(by="IoU", ascending=False)
df = df.drop(columns=["Unnamed: 0"])

df

,Experiment,Pixel Accuracy,IoU
1,Default U-Net,0.9390,0.7044
8,Only Middle Skip Connection,0.9391,0.6940
7,Only Top Skip Connection,0.9323,0.6912
10,Bottom Down Block Connect To Top Up Block,0.9224,0.6438
2,Extra Block U-Net,0.9161,0.6349
3,U-Net With No Skip Connections,0.9006,0.5852
6,Only Bottom Skip Connection,0.9002,0.5833
11,Top Down Block Connect To Bottom Up Block,0.9005,0.5833
9,Only Two Blocks,0.9259,0.5465
0,Top Skip Connection Removed,0.9167,0.4570


In [14]:
df.to_csv("results_table_clean.csv", index=False)

In [19]:
print(df.to_markdown(index=False))

| Experiment                                |   Pixel Accuracy |    IoU |
|:------------------------------------------|-----------------:|-------:|
| Default U-Net                             |           0.939  | 0.7044 |
| Only Middle Skip Connection               |           0.9391 | 0.694  |
| Only Top Skip Connection                  |           0.9323 | 0.6912 |
| Bottom Down Block Connect To Top Up Block |           0.9224 | 0.6438 |
| Extra Block U-Net                         |           0.9161 | 0.6349 |
| U-Net With No Skip Connections            |           0.9006 | 0.5852 |
| Only Bottom Skip Connection               |           0.9002 | 0.5833 |
| Top Down Block Connect To Bottom Up Block |           0.9005 | 0.5833 |
| Only Two Blocks                           |           0.9259 | 0.5465 |
| Top Skip Connection Removed               |           0.9167 | 0.457  |
| Only One Block                            |           0.9127 | 0.4311 |
| Only One Block With More Capacity   

We use two main metrics to evaluate our U-Nets.

Pixel accuracy is the percentage of pixels that the classifier got correct. This metric can be helpful, but if the dataset is heavily skewed towards postive or negative labels then it is less insightful.

Intersection over Union (IoU) is the area of intersection where the classifier predicted a postive label and the ground truth was postive divided by the union of these two areas. 1 is a perfect score for this metric. 

We see that all of the models have similar pixel-level accuracy. This is expected since the dataset is heavily imbalanced toward non-cancer pixels. Because of this, even small changes in accuracy are not very informative. IoU is where the models actually start to separate.

The standard U-Net performs the best overall, with an IoU of 0.7044. Interestingly, removing skip connections has only a small effect when keeping either the top or middle skip connection, dropping IoU by about 0.01. However, when only the lowest skip connection is used, performance drops significantly to an IoU of 0.5833.

We also tested changing the capacity of the U-Net by adding or removing blocks. The four-block U-Net performs decently with an IoU of 0.6349, while the smaller models perform noticeably worse overall.
Finally, we experimented with asymmetric skip connections, where the skip goes from the first down block to the first up block or vice versa. These do not behave the same. The bottom-to-top U-Net performs fairly well with an IoU of 0.6438, while the top-to-bottom version performs worse at 0.5833.

Overall, these experiments show that the top and middle skip connections are the most important for U-Net performance. Adding extra skip connections does not really help and only gives marginal improvements at best.

More details about the implementations and training setup can be found in the GitHub repo here: https://github.com/kurteh/deep_learning_522_project/tree/main/architecture_experiments